In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "mlp" / "combined"


# MLP Experience Replay (Combined)
Train an MLPClassifier with year-wise incremental scaling and a combined experience replay strategy.

In [ ]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

from src.mlp_replay.data import (
    load_dataset_and_splits,
    prepare_raw_features_for_year,
    prepare_features_for_year,
    precompute_yearly_raw_cache,
)
from src.mlp_replay.checkpointing import (
    create_empty_training_history,
    load_all_training_histories,
    save_all_training_histories,
    load_completion_status,
    save_completion_status,
    append_training_log,
    format_ratio_key,
    get_cached_raw_year,
)

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [ ]:
ds_path = PROJECT_ROOT / 'training_data_with_features_plus_monthly_indices.zarr'
split_path = PROJECT_ROOT / 'data_split.npz'
ds, train_pixel_indices, val_pixel_indices, test_pixel_indices = load_dataset_and_splits(ds_path, split_path)


## 2. Feature Engineering

`prepare_raw_features_for_year` and `prepare_features_for_year` now live in `src/mlp_replay/data.py` and are imported above.


## 3. Initialize Class Weights and MLP

In [ ]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)


train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train')
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation')

print('Computing class weights from cached training labels...')
train_label_batches = [y_batch for _, y_batch in train_feature_cache.values() if len(y_batch) > 0]
if not train_label_batches:
    raise ValueError('No valid training labels after filtering.')

all_train_labels = np.concatenate(train_label_batches).astype(int)
all_train_labels = all_train_labels[np.isin(all_train_labels, [0, 1])]
if len(all_train_labels) == 0:
    raise ValueError('No valid training labels after filtering.')

classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print('Class weights:')
print(f"  Class 0: {class_weight_dict[0]:.4f}")
print(f"  Class 1: {class_weight_dict[1]:.4f}")

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

## 4. Replay-Enabled Online Training

### Replay Sampling Strategy (Combined Buffers + Random Remainder)
Replay is allocated **once per training year** into configurable buffers:
- **HARD_EXAMPLE**
- **CONFIDENTLY_CORRECT**
- **UNCERTAINITY_PRIORITIZATION**
- **POSITIVE_RATE = (buffer_fraction, target_positive_percent)**
- **Random remainder**

For current-year sample size $N$ and replay ratio $RR$, the strategy targets are absolute-by-year:
$$B_{HE}=\lfloor N\cdot HE\rfloor,\; B_{CC}=\lfloor N\cdot CC\rfloor,\; B_{UP}=\lfloor N\cdot UP\rfloor,\; B_{PR}=\lfloor N\cdot PR_{buf}\rfloor$$
with feasibility constraint $HE + CC + UP + PR_{buf} \le RR$.

All non-random strategies are sampled first with exclusion to avoid overlap. The remaining budget is filled uniformly at random from unused replay candidates.

Sampling details:
1. **Hard Example**: prioritize misclassified samples using confidence of error.
2. **Confidently Correct**: prioritize correctly classified samples using confidence of correctness.
3. **Uncertainty Prioritization**: Gaussian weighting around per-year optimal F1 threshold.
4. **Positive Rate**: enforce requested positive fraction inside its own buffer, with graceful fallback when pool constraints prevent exact matching.

Each replay buffer gets its own class weights derived from that year's sampled labels, and each buffer's weights are normalized to mean 1.0 before training.

In [ ]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.5]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = 42

# Combined strategy fractions in [0, 1].
HARD_EXAMPLE = 0
CONFIDENTLY_CORRECT = 0.2
UNCERTAINITY_PRIORITIZATION = 0.1
POSITIVE_RATE = (0.2, 10)  # (buffer_fraction, target_positive_percent)
MISCLASSIFICATION_BUFFER = 0.0  # Fraction of budget for misclassification buffer
REPLAY_WEIGHT_SCALE = 1.0  # Controls the relative weight/influence of replay samples vs current-year samples

UNCERTAINTY_GAUSSIAN_STD = 0.03
UNCERTAINTY_THRESHOLD_GRID = np.linspace(0.0, 1.0, 201)
EPSILON = 1e-12

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals():
    raise ValueError('Raw feature caches not found. Run the precompute cell first.')

def _format_float_token(value, decimals=3):
    token = f'{float(value):.{decimals}f}'
    token = token.rstrip('0').rstrip('.')
    return token if token else '0'

def _sanitize_for_windows_filename(text):
    # Windows forbidden chars: <>:"/\\|?*
    forbidden = '<>:"/\\|?*'
    out = ''.join('-' if ch in forbidden else ch for ch in str(text))
    return out.strip(' .')

pr_buffer_fraction = float(POSITIVE_RATE[0])
pr_target_percent = float(POSITIVE_RATE[1])
pr_target_rate = pr_target_percent / 100.0

if not (0.0 <= HARD_EXAMPLE <= 1.0):
    raise ValueError('HARD_EXAMPLE must be in [0, 1].')
if not (0.0 <= CONFIDENTLY_CORRECT <= 1.0):
    raise ValueError('CONFIDENTLY_CORRECT must be in [0, 1].')
if not (0.0 <= UNCERTAINITY_PRIORITIZATION <= 1.0):
    raise ValueError('UNCERTAINITY_PRIORITIZATION must be in [0, 1].')
if not (0.0 <= pr_buffer_fraction <= 1.0):
    raise ValueError('POSITIVE_RATE buffer fraction must be in [0, 1].')
if not (0.0 <= pr_target_percent <= 100.0):
    raise ValueError('POSITIVE_RATE target percent must be in [0, 100].')
if not (0.0 <= MISCLASSIFICATION_BUFFER <= 1.0):
    raise ValueError('MISCLASSIFICATION_BUFFER must be in [0, 1].')

strategy_absolute_sum = HARD_EXAMPLE + CONFIDENTLY_CORRECT + UNCERTAINITY_PRIORITIZATION + pr_buffer_fraction + MISCLASSIFICATION_BUFFER
invalid_rr = [rr for rr in REPLAY_RATIOS if strategy_absolute_sum > rr + 1e-12]
if invalid_rr:
    raise ValueError(
        f'Absolute strategy sum must be <= RR. Got sum={strategy_absolute_sum:.6f}, '
        f'invalid RR values={invalid_rr}. Reduce HARD_EXAMPLE/CONFIDENTLY_CORRECT/'
        'UNCERTAINITY_PRIORITIZATION/POSITIVE_RATE/MISCLASSIFICATION_BUFFER or increase RR.'
    )

def build_strategy_suffix():
    token = (
        f'_combined_HE={_format_float_token(HARD_EXAMPLE)}'
        f'_CC={_format_float_token(CONFIDENTLY_CORRECT)}'
        f'_UP={_format_float_token(UNCERTAINITY_PRIORITIZATION)}'
        f'_PR=({_format_float_token(pr_buffer_fraction)},{_format_float_token(pr_target_percent, decimals=1)})'
        f'_MC={_format_float_token(MISCLASSIFICATION_BUFFER)}'
        f'_RWS={_format_float_token(REPLAY_WEIGHT_SCALE)}'
    )
    return _sanitize_for_windows_filename(token)

def build_ratio_suffix(replay_ratio):
    token = (
        f'{build_strategy_suffix()}'
        f'_RR={_format_float_token(replay_ratio)}'
    )
    return _sanitize_for_windows_filename(token)

def build_mlp_model():
    return MLPClassifier(
        hidden_layer_sizes=(64,),
        activation='relu',
        alpha=0.0001,
        random_state=42,
        solver='adam',
        learning_rate='adaptive',
        max_iter=1,
        learning_rate_init=0.001,
        warm_start=False,
        verbose=False,
    )

def compute_optimal_f1_threshold(y_true, y_proba, default_threshold=0.5):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float(default_threshold)

    from sklearn.metrics import precision_recall_curve

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = int(np.argmax(f1_scores))
    if best_idx < len(thresholds):
        return float(thresholds[best_idx])
    return float(default_threshold)

def weighted_choice_without_replacement(indices, weights, sample_size, rng):
    if sample_size <= 0 or len(indices) == 0:
        return np.empty((0,), dtype=np.int64)

    sample_size = min(int(sample_size), len(indices))
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.where(np.isnan(weights) | (weights < 0), 0.0, weights)
    total_weight = float(weights.sum())

    if total_weight <= 0:
        selected_positions = rng.choice(len(indices), size=sample_size, replace=False)
    else:
        non_zero_mask = weights > 0
        n_non_zero = int(np.sum(non_zero_mask))
        if n_non_zero < sample_size:
            chosen_non_zero = np.where(non_zero_mask)[0]
            remaining_needed = sample_size - n_non_zero
            zero_indices = np.where(~non_zero_mask)[0]
            chosen_zero = rng.choice(zero_indices, size=remaining_needed, replace=False)
            selected_positions = np.concatenate([chosen_non_zero, chosen_zero])
            rng.shuffle(selected_positions)
        else:
            prob = weights / total_weight
            selected_positions = rng.choice(len(indices), size=sample_size, replace=False, p=prob)

    return np.asarray(indices, dtype=np.int64)[selected_positions]

def allocate_per_year_targets(replay_year_spans, target_size):
    if target_size <= 0 or len(replay_year_spans) == 0:
        return []

    n_years_with_data = len(replay_year_spans)
    base_quota = target_size // n_years_with_data
    remainder = target_size % n_years_with_data

    per_year = []
    for idx, year_span in enumerate(replay_year_spans):
        year_idx, start, end = year_span
        requested = base_quota + (1 if idx < remainder else 0)
        year_size = end - start
        per_year.append((year_idx, start, end, min(requested, year_size)))
    return per_year

def top_up_indices(target_size, selected_indices, total_pool_size, rng):
    selected_indices = np.asarray(selected_indices, dtype=np.int64)
    shortfall = int(target_size) - len(selected_indices)
    if shortfall <= 0:
        return selected_indices

    all_indices = np.arange(total_pool_size, dtype=np.int64)
    remaining_indices = np.setdiff1d(all_indices, selected_indices, assume_unique=False)
    if len(remaining_indices) == 0:
        return selected_indices

    top_up = rng.choice(remaining_indices, size=min(shortfall, len(remaining_indices)), replace=False)
    return np.concatenate([selected_indices, top_up.astype(np.int64, copy=False)])

def sample_hard_replay_indices(model, X_replay_pool, y_replay_pool, replay_year_spans, hard_target_size, rng):
    if hard_target_size <= 0 or len(replay_year_spans) == 0:
        return np.empty((0,), dtype=np.int64)

    selected = []
    for _, start, end, target in allocate_per_year_targets(replay_year_spans, hard_target_size):
        if target <= 0:
            continue
        year_indices = np.arange(start, end, dtype=np.int64)
        y_year = y_replay_pool[start:end]
        y_proba_year = model.predict_proba(X_replay_pool[start:end])[:, 1]
        threshold = compute_optimal_f1_threshold(y_year, y_proba_year, default_threshold=0.5)
        y_pred_year = (y_proba_year >= threshold).astype(np.int64)
        hard_mask = y_pred_year != y_year
        hard_weights = np.where(hard_mask, np.abs(y_year - y_proba_year), 0.0)
        chosen = weighted_choice_without_replacement(year_indices, hard_weights, target, rng)
        if len(chosen) > 0:
            selected.append(chosen)

    hard_indices = np.concatenate(selected).astype(np.int64, copy=False) if selected else np.empty((0,), dtype=np.int64)
    return top_up_indices(hard_target_size, hard_indices, len(y_replay_pool), rng)

def sample_confident_replay_indices(model, X_replay_pool, y_replay_pool, replay_year_spans, confident_target_size, rng):
    if confident_target_size <= 0 or len(replay_year_spans) == 0:
        return np.empty((0,), dtype=np.int64)

    selected = []
    for _, start, end, target in allocate_per_year_targets(replay_year_spans, confident_target_size):
        if target <= 0:
            continue
        year_indices = np.arange(start, end, dtype=np.int64)
        y_year = y_replay_pool[start:end]
        y_proba_year = model.predict_proba(X_replay_pool[start:end])[:, 1]
        threshold = compute_optimal_f1_threshold(y_year, y_proba_year, default_threshold=0.5)
        y_pred_year = (y_proba_year >= threshold).astype(np.int64)
        correct_mask = y_pred_year == y_year
        confident_weights = np.where(correct_mask, 1.0 - np.abs(y_year - y_proba_year), 0.0)
        chosen = weighted_choice_without_replacement(year_indices, confident_weights, target, rng)
        if len(chosen) > 0:
            selected.append(chosen)

    confident_indices = np.concatenate(selected).astype(np.int64, copy=False) if selected else np.empty((0,), dtype=np.int64)
    return top_up_indices(confident_target_size, confident_indices, len(y_replay_pool), rng)

def sample_uncertain_replay_indices(model, X_replay_pool, y_replay_pool, replay_year_spans, uncertain_target_size, rng, gaussian_std):
    if uncertain_target_size <= 0 or len(replay_year_spans) == 0:
        return np.empty((0,), dtype=np.int64), []

    selected = []
    threshold_values = []

    for _, start, end, target in allocate_per_year_targets(replay_year_spans, uncertain_target_size):
        if target <= 0:
            continue

        year_indices = np.arange(start, end, dtype=np.int64)
        y_year = y_replay_pool[start:end]
        y_proba_year = model.predict_proba(X_replay_pool[start:end])[:, 1]
        threshold = compute_optimal_f1_threshold(y_year, y_proba_year, default_threshold=0.5)
        threshold_values.append(threshold)

        distances = np.abs(y_proba_year - threshold)
        if len(distances) > 0 and target > 0:
            k_idx = min(int(target) - 1, len(distances) - 1)
            d_k = np.partition(distances, k_idx)[k_idx]
            std = max(float(gaussian_std), d_k / 4.8)
        else:
            std = float(gaussian_std)

        gaussian_weights = np.exp(-(distances ** 2) / (2.0 * (std ** 2) + EPSILON))
        chosen = weighted_choice_without_replacement(year_indices, gaussian_weights, target, rng)
        if len(chosen) > 0:
            selected.append(chosen)

    uncertain_indices = np.concatenate(selected).astype(np.int64, copy=False) if selected else np.empty((0,), dtype=np.int64)
    uncertain_indices = top_up_indices(uncertain_target_size, uncertain_indices, len(y_replay_pool), rng)
    return uncertain_indices, threshold_values

def sample_positive_rate_indices(y_replay_pool, target_size, target_positive_rate, excluded_indices, rng):
    if target_size <= 0 or len(y_replay_pool) == 0:
        return np.empty((0,), dtype=np.int64), np.nan

    all_indices = np.arange(len(y_replay_pool), dtype=np.int64)
    available_indices = np.setdiff1d(all_indices, np.asarray(excluded_indices, dtype=np.int64), assume_unique=False)
    if len(available_indices) == 0:
        return np.empty((0,), dtype=np.int64), np.nan

    positive_indices = available_indices[y_replay_pool[available_indices] == 1]
    negative_indices = available_indices[y_replay_pool[available_indices] == 0]

    desired_pos = int(np.round(target_size * float(target_positive_rate)))
    take_pos = min(desired_pos, len(positive_indices))
    take_neg = min(int(target_size) - take_pos, len(negative_indices))

    chosen_pos = rng.choice(positive_indices, size=take_pos, replace=False).astype(np.int64, copy=False) if take_pos > 0 else np.empty((0,), dtype=np.int64)
    chosen_neg = rng.choice(negative_indices, size=take_neg, replace=False).astype(np.int64, copy=False) if take_neg > 0 else np.empty((0,), dtype=np.int64)

    selected = np.concatenate([chosen_pos, chosen_neg])

    shortfall = int(target_size) - len(selected)
    if shortfall > 0:
        remaining = np.setdiff1d(available_indices, selected, assume_unique=False)
        if len(remaining) > 0:
            top_up = rng.choice(remaining, size=min(shortfall, len(remaining)), replace=False)
            selected = np.concatenate([selected, top_up.astype(np.int64, copy=False)])

    achieved_percent = float(100.0 * np.mean(y_replay_pool[selected])) if len(selected) > 0 else np.nan
    return selected.astype(np.int64, copy=False), achieved_percent

def sample_random_replay_indices(replay_pool_size, random_target_size, excluded_indices, rng):
    if random_target_size <= 0 or replay_pool_size <= 0:
        return np.empty((0,), dtype=np.int64)

    all_indices = np.arange(replay_pool_size, dtype=np.int64)
    available_indices = np.setdiff1d(all_indices, np.asarray(excluded_indices, dtype=np.int64), assume_unique=False)
    if len(available_indices) == 0:
        return np.empty((0,), dtype=np.int64)

    take = min(int(random_target_size), len(available_indices))
    return rng.choice(available_indices, size=take, replace=False).astype(np.int64, copy=False)

def compute_binary_weight_map(labels):
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return {0: 1.0, 1: 1.0}

    unique_labels = np.unique(labels)
    if len(unique_labels) >= 2:
        weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=labels)
        return {0: float(weights[0]), 1: float(weights[1])}

    # Smoothed fallback for one-class segments.
    n = float(len(labels))
    pos_prob = (float(np.sum(labels == 1)) + 1.0) / (n + 2.0)
    neg_prob = 1.0 - pos_prob
    return {
        0: float(1.0 / max(2.0 * neg_prob, 1e-12)),
        1: float(1.0 / max(2.0 * pos_prob, 1e-12)),
    }

def build_normalized_weights(labels, weight_map):
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return np.empty((0,), dtype=np.float64)

    weights = np.array([weight_map[int(lbl)] for lbl in labels], dtype=np.float64)
    mean_weight = float(np.mean(weights))
    if mean_weight <= 0 or np.isnan(mean_weight):
        return np.ones_like(weights, dtype=np.float64)
    return weights / mean_weight

MISCLASS_SNAPSHOT_TEMPLATE = 'mlp_replay_misclass_snapshot_{run_key}_year_{year}.pkl'

def get_misclass_snapshot_path(checkpoint_dir, run_key, year_val):
    filename = MISCLASS_SNAPSHOT_TEMPLATE.format(run_key=run_key, year=year_val)
    return checkpoint_dir / filename

def save_misclass_snapshot(path, misclass_local_indices):
    serializable = {}
    for past_year_idx, idx_array in misclass_local_indices.items():
        serializable[int(past_year_idx)] = np.asarray(idx_array, dtype=np.int64)
    with open(path, 'wb') as f:
        pickle.dump(serializable, f)

def load_misclass_snapshot(path):
    if not path.exists():
        return {}
    with open(path, 'rb') as f:
        data = pickle.load(f)
    if not isinstance(data, dict):
        return {}

    cleaned = {}
    for key, value in data.items():
        cleaned[int(key)] = np.asarray(value, dtype=np.int64)
    return cleaned

def compute_misclassified_local_indices(model, scaler, train_cache, max_past_year_idx):
    misclass_local = {}
    for past_year_idx in range(1, max_past_year_idx + 1):
        X_past_raw, y_past = get_cached_raw_year(train_cache, past_year_idx)
        if len(X_past_raw) == 0:
            continue

        X_past_scaled = scaler.transform(X_past_raw)
        y_past_pred = model.predict(X_past_scaled)
        misclass_idx = np.flatnonzero(y_past_pred != y_past)
        misclass_local[past_year_idx] = misclass_idx.astype(np.int64, copy=False)

    return misclass_local

def build_misclass_pool_indices(misclass_local_indices, replay_year_spans):
    misclass_global = []
    for past_year_idx, start, end in replay_year_spans:
        n_samples = end - start
        local_idx = misclass_local_indices.get(past_year_idx)
        if local_idx is None or len(local_idx) == 0:
            continue

        valid_local = local_idx[(local_idx >= 0) & (local_idx < n_samples)]
        if len(valid_local) == 0:
            continue

        misclass_global.append(start + valid_local)

    if not misclass_global:
        return np.empty((0,), dtype=np.int64)

    return np.unique(np.concatenate(misclass_global)).astype(np.int64, copy=False)

def sample_misclass_replay_indices(misclass_pool_indices, misclass_target_size, excluded_indices, rng):
    if misclass_target_size <= 0 or len(misclass_pool_indices) == 0:
        return np.empty((0,), dtype=np.int64)

    available_misclass = np.setdiff1d(misclass_pool_indices, np.asarray(excluded_indices, dtype=np.int64), assume_unique=False)
    if len(available_misclass) == 0:
        return np.empty((0,), dtype=np.int64)

    misclass_take = min(misclass_target_size, len(available_misclass))
    selected = rng.choice(available_misclass, size=misclass_take, replace=False)
    return selected.astype(np.int64, copy=False)

STRATEGY_SUFFIX = build_strategy_suffix()
CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_experience_replay_combined'
CHECKPOINT_DIR.mkdir(exist_ok=True)

ALL_HISTORIES_FILE = CHECKPOINT_DIR / f'mlp_replay_all_training_histories{STRATEGY_SUFFIX}.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / f'mlp_replay_completion_status{STRATEGY_SUFFIX}.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / f'mlp_replay_training_log{STRATEGY_SUFFIX}.txt'
COMBINED_HISTORY_FILENAME = f'mlp_classifier_history_prevyears_monthly_features{STRATEGY_SUFFIX}_all_ratios.csv'

all_training_histories = load_all_training_histories(ALL_HISTORIES_FILE)
completion_status = load_completion_status(COMPLETION_STATUS_FILE)

n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
print(
    'Strategy config: '
    f'HE={HARD_EXAMPLE}, CC={CONFIDENTLY_CORRECT}, UP={UNCERTAINITY_PRIORITIZATION}, '
    f'PR={POSITIVE_RATE}, absolute_sum={strategy_absolute_sum:.4f}'
)
append_training_log(
    TRAINING_LOG_FILE,
    'Started training run with config '
    f'HE={HARD_EXAMPLE}, CC={CONFIDENTLY_CORRECT}, UP={UNCERTAINITY_PRIORITIZATION}, '
    f'PR={POSITIVE_RATE}, ratios={REPLAY_RATIOS}',
)

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    ratio_suffix = build_ratio_suffix(replay_ratio)
    run_key = f'{ratio_key}{ratio_suffix}'

    models_dir_name = f'models_mlp{ratio_suffix}'
    scaler_file_template = f'scaler_year_{{year}}{ratio_suffix}.pkl'
    model_file_template = f'model_year_{{year}}{ratio_suffix}.pkl'
    final_scaler_filename = f'scaler_final{ratio_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model{ratio_suffix}.pkl'
    history_filename = f'mlp_classifier_history{ratio_suffix}.csv'

    models_dir = EXPERIMENTS_DIR / models_dir_name
    models_dir.mkdir(exist_ok=True)

    if run_key in completion_status.get('completed_ratios', []):
        print(f'[{run_key}] already completed. Skipping.')
        append_training_log(TRAINING_LOG_FILE, f'[{run_key}] skipped (already completed).')
        continue

    training_history = all_training_histories.get(run_key, create_empty_training_history(extra_keys=['hard_target_size', 'hard_used_size', 'confident_target_size', 'confident_used_size', 'uncertain_target_size', 'uncertain_used_size', 'positive_rate_target_size', 'positive_rate_used_size', 'positive_rate_target_percent', 'positive_rate_achieved_percent', 'misclass_target_size', 'misclass_used_size', 'random_target_size', 'random_used_size', 'uncertain_threshold_mean', 'wmean_hard', 'wmean_confident', 'wmean_uncertain', 'wmean_positive_rate', 'wmean_misclass', 'wmean_random', 'artifact_suffix']))
    for key in create_empty_training_history(extra_keys=['hard_target_size', 'hard_used_size', 'confident_target_size', 'confident_used_size', 'uncertain_target_size', 'uncertain_used_size', 'positive_rate_target_size', 'positive_rate_used_size', 'positive_rate_target_percent', 'positive_rate_achieved_percent', 'misclass_target_size', 'misclass_used_size', 'random_target_size', 'random_used_size', 'uncertain_threshold_mean', 'wmean_hard', 'wmean_confident', 'wmean_uncertain', 'wmean_positive_rate', 'wmean_misclass', 'wmean_random', 'artifact_suffix']).keys():
        training_history.setdefault(key, [])

    completed_years = set(int(y) for y in completion_status.get('completed_years', {}).get(run_key, []))
    completed_years.update(int(y) for y in training_history.get('year', []))
    completion_status.setdefault('completed_years', {})[run_key] = sorted(list(completed_years))

    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model = None
    incremental_scaler = None
    start_year_idx = 1
    misclass_local_indices = {}

    if completed_years:
        resume_candidate_years = sorted(completed_years, reverse=True)
        resumed = False
        for resume_year in resume_candidate_years:
            year_model_path = models_dir / model_file_template.format(year=resume_year)
            year_scaler_path = models_dir / scaler_file_template.format(year=resume_year)
            if year_model_path.exists() and year_scaler_path.exists() and resume_year in year_value_to_idx:
                with open(year_model_path, 'rb') as f:
                    model = pickle.load(f)
                with open(year_scaler_path, 'rb') as f:
                    incremental_scaler = pickle.load(f)
                start_year_idx = year_value_to_idx[resume_year] + 1
                resumed = True
                print(f'[{run_key}] resuming from year {resume_year}; next index {start_year_idx}.')
                append_training_log(TRAINING_LOG_FILE, f'[{run_key}] resumed from year {resume_year}.')
                break

        if resumed:
            completed_years = {y for y in completed_years if y <= resume_year}
            if training_history.get('year'):
                keep_indices = [i for i, y in enumerate(training_history['year']) if y <= resume_year]
                for key in training_history.keys():
                    if isinstance(training_history[key], list):
                        training_history[key] = [training_history[key][i] for i in keep_indices]
            misclass_snapshot_path = get_misclass_snapshot_path(CHECKPOINT_DIR, run_key, resume_year)
            misclass_local_indices = load_misclass_snapshot(misclass_snapshot_path)
        else:
            print(f'[{run_key}] checkpoint artifacts inconsistent. Restarting this run key.')
            append_training_log(TRAINING_LOG_FILE, f'[{run_key}] restart due to inconsistent checkpoints.')
            completed_years = set()
            completion_status['completed_years'][run_key] = []
            training_history = create_empty_training_history(extra_keys=['hard_target_size', 'hard_used_size', 'confident_target_size', 'confident_used_size', 'uncertain_target_size', 'uncertain_used_size', 'positive_rate_target_size', 'positive_rate_used_size', 'positive_rate_target_percent', 'positive_rate_achieved_percent', 'misclass_target_size', 'misclass_used_size', 'random_target_size', 'random_used_size', 'uncertain_threshold_mean', 'wmean_hard', 'wmean_confident', 'wmean_uncertain', 'wmean_positive_rate', 'wmean_misclass', 'wmean_random', 'artifact_suffix'])
            misclass_local_indices = {}

    if model is None:
        model = build_mlp_model()
    if incremental_scaler is None:
        incremental_scaler = StandardScaler()

    print(f'[{run_key}] Output directory: {models_dir.resolve()}')
    print(f'[{run_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{run_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        X_train_raw, y_train_batch = get_cached_raw_year(train_feature_cache, year_idx)
        X_val_raw, y_val_batch = get_cached_raw_year(val_feature_cache, year_idx)

        if len(X_train_raw) == 0 or len(X_val_raw) == 0:
            print(f'[{run_key}] Year {year_val}: skipped (empty after filtering)')
            completed_years.add(year_val)
            completion_status['completed_years'][run_key] = sorted(list(completed_years))
            save_completion_status(COMPLETION_STATUS_FILE, completion_status)
            append_training_log(TRAINING_LOG_FILE, f'[{run_key}] year {year_val} skipped (empty).')
            continue

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        replay_X_parts = []
        replay_y_parts = []
        replay_year_spans = []
        replay_cursor = 0

        if REPLAY_ENABLED and year_idx > 1:
            for past_year_idx in range(1, year_idx):
                X_past_raw, y_past = get_cached_raw_year(train_feature_cache, past_year_idx)
                if len(X_past_raw) > 0:
                    X_past = incremental_scaler.transform(X_past_raw)
                    replay_X_parts.append(X_past)
                    replay_y_parts.append(y_past)
                    span_len = len(y_past)
                    replay_year_spans.append((past_year_idx, replay_cursor, replay_cursor + span_len))
                    replay_cursor += span_len

        if replay_X_parts:
            X_replay_pool = np.vstack(replay_X_parts)
            y_replay_pool = np.concatenate(replay_y_parts)
        else:
            X_replay_pool = np.empty((0, X_train_batch.shape[1]), dtype=X_train_batch.dtype)
            y_replay_pool = np.empty((0,), dtype=y_train_batch.dtype)

        replay_pool_size = len(y_replay_pool)
        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        # One-time yearly sampling for replay segments (no resampling per epoch).
        hard_target_size = int(np.floor(n_samples * HARD_EXAMPLE))
        confident_target_size = int(np.floor(n_samples * CONFIDENTLY_CORRECT))
        uncertain_target_size = int(np.floor(n_samples * UNCERTAINITY_PRIORITIZATION))
        positive_rate_target_size = int(np.floor(n_samples * pr_buffer_fraction))
        misclass_target_size = int(np.floor(n_samples * MISCLASSIFICATION_BUFFER))

        hard_indices = np.empty((0,), dtype=np.int64)
        confident_indices = np.empty((0,), dtype=np.int64)
        uncertain_indices = np.empty((0,), dtype=np.int64)
        positive_rate_indices = np.empty((0,), dtype=np.int64)
        misclass_indices = np.empty((0,), dtype=np.int64)
        random_indices = np.empty((0,), dtype=np.int64)

        uncertain_threshold_mean = np.nan
        positive_rate_achieved_percent = np.nan

        if replay_used_size > 0:
            hard_indices = sample_hard_replay_indices(
                model=model,
                X_replay_pool=X_replay_pool,
                y_replay_pool=y_replay_pool,
                replay_year_spans=replay_year_spans,
                hard_target_size=hard_target_size,
                rng=replay_rng,
            )

            confident_indices = sample_confident_replay_indices(
                model=model,
                X_replay_pool=X_replay_pool,
                y_replay_pool=y_replay_pool,
                replay_year_spans=replay_year_spans,
                confident_target_size=confident_target_size,
                rng=replay_rng,
            )
            confident_indices = np.setdiff1d(confident_indices, hard_indices, assume_unique=False)

            uncertain_indices, threshold_values = sample_uncertain_replay_indices(
                model=model,
                X_replay_pool=X_replay_pool,
                y_replay_pool=y_replay_pool,
                replay_year_spans=replay_year_spans,
                uncertain_target_size=uncertain_target_size,
                rng=replay_rng,
                gaussian_std=UNCERTAINTY_GAUSSIAN_STD,
            )
            exclude_before_uncertain = np.concatenate([hard_indices, confident_indices])
            uncertain_indices = np.setdiff1d(uncertain_indices, exclude_before_uncertain, assume_unique=False)
            if threshold_values:
                uncertain_threshold_mean = float(np.mean(threshold_values))

            exclude_before_pr = np.concatenate([hard_indices, confident_indices, uncertain_indices])
            positive_rate_indices, positive_rate_achieved_percent = sample_positive_rate_indices(
                y_replay_pool=y_replay_pool,
                target_size=positive_rate_target_size,
                target_positive_rate=pr_target_rate,
                excluded_indices=exclude_before_pr,
                rng=replay_rng,
            )

            exclude_before_misclass = np.concatenate([hard_indices, confident_indices, uncertain_indices, positive_rate_indices])
            misclass_pool_indices = build_misclass_pool_indices(misclass_local_indices, replay_year_spans)
            misclass_indices = sample_misclass_replay_indices(
                misclass_pool_indices=misclass_pool_indices,
                misclass_target_size=misclass_target_size,
                excluded_indices=exclude_before_misclass,
                rng=replay_rng,
            )

            selected_non_random = np.concatenate([hard_indices, confident_indices, uncertain_indices, positive_rate_indices, misclass_indices])
            random_target_size = max(0, int(replay_used_size - len(selected_non_random)))
            random_indices = sample_random_replay_indices(
                replay_pool_size=replay_pool_size,
                random_target_size=random_target_size,
                excluded_indices=selected_non_random,
                rng=replay_rng,
            )
        else:
            random_target_size = 0

        hard_used_size = int(len(hard_indices))
        confident_used_size = int(len(confident_indices))
        uncertain_used_size = int(len(uncertain_indices))
        positive_rate_used_size = int(len(positive_rate_indices))
        misclass_used_size = int(len(misclass_indices))
        random_used_size = int(len(random_indices))

        segment_order = [
            ('hard', hard_indices),
            ('confident', confident_indices),
            ('uncertain', uncertain_indices),
            ('positive_rate', positive_rate_indices),
            ('misclass', misclass_indices),
            ('random', random_indices),
        ]

        replay_indices_parts = [idx for _, idx in segment_order if len(idx) > 0]
        if replay_indices_parts:
            replay_indices = np.concatenate(replay_indices_parts).astype(np.int64, copy=False)
        else:
            replay_indices = np.empty((0,), dtype=np.int64)

        # Safety check: merged replay indices must be unique.
        if len(replay_indices) != len(np.unique(replay_indices)):
            raise ValueError(f'[{run_key}] Duplicate replay indices detected at year {year_val}.')

        wmean_by_segment = {
            'hard': np.nan,
            'confident': np.nan,
            'uncertain': np.nan,
            'positive_rate': np.nan,
            'misclass': np.nan,
            'random': np.nan,
        }

        current_weight_map = compute_binary_weight_map(y_train_batch)
        current_weights = build_normalized_weights(y_train_batch, current_weight_map)

        if len(replay_indices) > 0:
            X_replay_sampled = X_replay_pool[replay_indices]
            y_replay_sampled = y_replay_pool[replay_indices]

            replay_weight_parts = []
            for segment_name, segment_indices in segment_order:
                if len(segment_indices) == 0:
                    continue
                y_segment = y_replay_pool[segment_indices]
                weight_map = compute_binary_weight_map(y_segment)
                w_segment = build_normalized_weights(y_segment, weight_map)
                wmean_by_segment[segment_name] = float(np.mean(w_segment)) if len(w_segment) > 0 else np.nan
                replay_weight_parts.append(w_segment)

            replay_weights = np.concatenate(replay_weight_parts) if replay_weight_parts else np.empty((0,), dtype=np.float64)

            X_combined_base = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
            y_combined_base = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
            sample_weights_base = np.concatenate([current_weights, replay_weights * REPLAY_WEIGHT_SCALE], axis=0)
        else:
            X_combined_base = X_train_batch
            y_combined_base = y_train_batch
            sample_weights_base = current_weights

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None

        for _ in range(MAX_EPOCHS):
            combined_n_samples = len(X_combined_base)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined_base[shuffle_idx]
            y_train_shuffled = y_combined_base[shuffle_idx]
            w_train_shuffled = sample_weights_base[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                w_chunk = w_train_shuffled[start_idx:end_idx]
                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=w_chunk)

            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = {
                    'coefs': [w.copy() for w in model.coefs_],
                    'intercepts': [b.copy() for b in model.intercepts_],
                    'n_layers_': model.n_layers_,
                    'n_outputs_': getattr(model, 'n_outputs_', None),
                    'out_activation_': getattr(model, 'out_activation_', None),
                    # [NEW] Save optimizer state
                    'optimizer_state': {
                        'type': 'adam',
                        'ms': [m.copy() for m in model._optimizer.ms],
                        'vs': [v.copy() for v in model._optimizer.vs],
                        't': model._optimizer.t
                    } if hasattr(model, '_optimizer') and model._optimizer is not None and hasattr(model._optimizer, 'ms') else (
                        {
                            'type': 'sgd',
                            'velocities': [v.copy() for v in model._optimizer.velocities]
                        } if hasattr(model, '_optimizer') and model._optimizer is not None and hasattr(model._optimizer, 'velocities') else None
                    ),
                }
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    if best_model_state is not None:
                        model.coefs_ = [w.copy() for w in best_model_state['coefs']]
                        model.intercepts_ = [b.copy() for b in best_model_state['intercepts']]
                        model.n_layers_ = best_model_state['n_layers_']
                        if best_model_state['n_outputs_'] is not None:
                            model.n_outputs_ = best_model_state['n_outputs_']
                        if best_model_state['out_activation_'] is not None:
                            model.out_activation_ = best_model_state['out_activation_']
                        
                        # [NEW] Restore optimizer state
                        opt_state = best_model_state.get('optimizer_state')
                        if opt_state is not None and hasattr(model, '_optimizer') and model._optimizer is not None:
                            opt = model._optimizer
                            if opt_state['type'] == 'adam' and hasattr(opt, 'ms'):
                                opt.ms = [m.copy() for m in opt_state['ms']]
                                opt.vs = [v.copy() for v in opt_state['vs']]
                                opt.t = opt_state['t']
                            elif opt_state['type'] == 'sgd' and hasattr(opt, 'velocities'):
                                opt.velocities = [v.copy() for v in opt_state['velocities']]
                    break

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))
        training_history['hard_target_size'].append(int(hard_target_size))
        training_history['hard_used_size'].append(int(hard_used_size))
        training_history['confident_target_size'].append(int(confident_target_size))
        training_history['confident_used_size'].append(int(confident_used_size))
        training_history['uncertain_target_size'].append(int(uncertain_target_size))
        training_history['uncertain_used_size'].append(int(uncertain_used_size))
        training_history['positive_rate_target_size'].append(int(positive_rate_target_size))
        training_history['positive_rate_used_size'].append(int(positive_rate_used_size))
        training_history['positive_rate_target_percent'].append(float(pr_target_percent))
        training_history['positive_rate_achieved_percent'].append(float(positive_rate_achieved_percent) if not np.isnan(positive_rate_achieved_percent) else np.nan)
        training_history['misclass_target_size'].append(int(misclass_target_size))
        training_history['misclass_used_size'].append(int(misclass_used_size))
        training_history['random_target_size'].append(int(max(0, replay_used_size - (hard_used_size + confident_used_size + uncertain_used_size + positive_rate_used_size + misclass_used_size))))
        training_history['random_used_size'].append(int(random_used_size))
        training_history['uncertain_threshold_mean'].append(float(uncertain_threshold_mean) if not np.isnan(uncertain_threshold_mean) else np.nan)
        training_history['wmean_hard'].append(float(wmean_by_segment['hard']) if not np.isnan(wmean_by_segment['hard']) else np.nan)
        training_history['wmean_confident'].append(float(wmean_by_segment['confident']) if not np.isnan(wmean_by_segment['confident']) else np.nan)
        training_history['wmean_uncertain'].append(float(wmean_by_segment['uncertain']) if not np.isnan(wmean_by_segment['uncertain']) else np.nan)
        training_history['wmean_positive_rate'].append(float(wmean_by_segment['positive_rate']) if not np.isnan(wmean_by_segment['positive_rate']) else np.nan)
        training_history['wmean_misclass'].append(float(wmean_by_segment['misclass']) if not np.isnan(wmean_by_segment['misclass']) else np.nan)
        training_history['wmean_random'].append(float(wmean_by_segment['random']) if not np.isnan(wmean_by_segment['random']) else np.nan)
        training_history['artifact_suffix'].append(ratio_suffix)

        if REPLAY_ENABLED and year_idx >= 1 and MISCLASSIFICATION_BUFFER > 0:
            misclass_local_indices = compute_misclassified_local_indices(
                model,
                incremental_scaler,
                train_feature_cache,
                year_idx,
            )
            misclass_snapshot_path = get_misclass_snapshot_path(CHECKPOINT_DIR, run_key, year_val)
            save_misclass_snapshot(misclass_snapshot_path, misclass_local_indices)

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][run_key] = sorted(list(completed_years))
        all_training_histories[run_key] = training_history.copy()
        save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)

        print(
            f'[{run_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}, '
            f'HE={hard_used_size:,}, CC={confident_used_size:,}, UP={uncertain_used_size:,}, PR={positive_rate_used_size:,}, MC={misclass_used_size:,}, RND={random_used_size:,}'
        )
        append_training_log(
            TRAINING_LOG_FILE,
            f'[{run_key}] year {year_val} done: replay_used={replay_used_size}/{replay_pool_size}, '
            f'HE={hard_used_size}, CC={confident_used_size}, UP={uncertain_used_size}, '
            f'PR={positive_rate_used_size} (target={pr_target_percent}%, achieved={positive_rate_achieved_percent}), '
            f'MC={misclass_used_size}, RND={random_used_size}.',
        )

    if hasattr(incremental_scaler, 'mean_'):
        final_scaler_path = models_dir / final_scaler_filename
        with open(final_scaler_path, 'wb') as f:
            pickle.dump(incremental_scaler, f)
        print(f'[{run_key}] Final scaler saved: {final_scaler_path.name}')

    final_model_path = models_dir / final_model_filename
    with open(final_model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'[{run_key}] Final model saved: {final_model_path.name}')

    ratio_history_df = pd.DataFrame(training_history).sort_values('year').reset_index(drop=True)
    ratio_history_path = models_dir / history_filename
    ratio_history_df.to_csv(ratio_history_path, index=False)
    print(f'[{run_key}] History saved: {ratio_history_path}')

    if set(all_target_year_values).issubset(completed_years):
        if run_key not in completion_status.get('completed_ratios', []):
            completion_status.setdefault('completed_ratios', []).append(run_key)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)
        append_training_log(TRAINING_LOG_FILE, f'[{run_key}] marked as fully completed.')

    all_training_histories[run_key] = training_history.copy()
    save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)

combined_history_frames = []
for run_key, history_dict in all_training_histories.items():
    if not isinstance(history_dict, dict):
        continue
    if len(history_dict.get('year', [])) == 0:
        continue
    df_ratio = pd.DataFrame(history_dict).sort_values('year').reset_index(drop=True)
    df_ratio['run_key'] = run_key
    if '_RR=' in run_key:
        rr_part = run_key.split('_RR=')[-1]
        rr_value = rr_part.split('_')[0]
        try:
            df_ratio['replay_ratio'] = float(rr_value)
        except ValueError:
            df_ratio['replay_ratio'] = np.nan
    else:
        df_ratio['replay_ratio'] = np.nan
    combined_history_frames.append(df_ratio)

if combined_history_frames:
    combined_history_df = pd.concat(combined_history_frames, ignore_index=True)
    combined_history_df = combined_history_df.sort_values(['replay_ratio', 'year']).reset_index(drop=True)
    combined_history_path = EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME
    combined_history_df.to_csv(combined_history_path, index=False)
    print(f'Combined history saved: {combined_history_path}')
    print(f'Combined rows: {len(combined_history_df):,}')
else:
    print('No history data available to write combined history.')

save_completion_status(COMPLETION_STATUS_FILE, completion_status)
save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
append_training_log(TRAINING_LOG_FILE, 'Training run completed.')

## 5. Save Training History

In [19]:
import json

# Reuse strategy parameters from the training cell if they exist.
# Fallback defaults are used only when this cell is run standalone.
HARD_EXAMPLE = float(globals().get('HARD_EXAMPLE', 0.0))
CONFIDENTLY_CORRECT = float(globals().get('CONFIDENTLY_CORRECT', 0.2))
UNCERTAINITY_PRIORITIZATION = float(globals().get('UNCERTAINITY_PRIORITIZATION', 0.2))
POSITIVE_RATE = globals().get('POSITIVE_RATE', (0.1, 10))
MISCLASSIFICATION_BUFFER = float(globals().get('MISCLASSIFICATION_BUFFER', 0.0))
REPLAY_WEIGHT_SCALE = float(globals().get('REPLAY_WEIGHT_SCALE', 1.0))
if not isinstance(POSITIVE_RATE, (tuple, list)) or len(POSITIVE_RATE) != 2:
    raise ValueError("POSITIVE_RATE must be a 2-item tuple/list: (buffer_fraction, target_positive_percent).")
POSITIVE_RATE = (float(POSITIVE_RATE[0]), float(POSITIVE_RATE[1]))


def _format_float_token(value, decimals=3):
    token = f'{float(value):.{decimals}f}'
    token = token.rstrip('0').rstrip('.')
    return token if token else '0'


def _sanitize_for_windows_filename(text):
    forbidden = '<>:"/\\|?*'
    out = ''.join('-' if ch in forbidden else ch for ch in str(text))
    return out.strip(' .')


def build_strategy_suffix():
    pr_buffer_fraction = float(POSITIVE_RATE[0])
    pr_target_percent = float(POSITIVE_RATE[1])
    token = (
        f'_combined_HE={_format_float_token(HARD_EXAMPLE)}'
        f'_CC={_format_float_token(CONFIDENTLY_CORRECT)}'
        f'_UP={_format_float_token(UNCERTAINITY_PRIORITIZATION)}'
        f'_PR=({_format_float_token(pr_buffer_fraction)},{_format_float_token(pr_target_percent, decimals=1)})'
        f'_MC={_format_float_token(MISCLASSIFICATION_BUFFER)}'
        f'_RWS={_format_float_token(REPLAY_WEIGHT_SCALE)}'
    )
    return _sanitize_for_windows_filename(token)


STRATEGY_SUFFIX = build_strategy_suffix()
CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_experience_replay_combined'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / f'mlp_replay_all_training_histories{STRATEGY_SUFFIX}.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / f'mlp_replay_completion_status{STRATEGY_SUFFIX}.json'
COMBINED_HISTORY_FILENAME = f'mlp_classifier_history_prevyears_monthly_features{STRATEGY_SUFFIX}_all_ratios.csv'

if ALL_HISTORIES_FILE.exists():
    with open(ALL_HISTORIES_FILE, 'rb') as f:
        all_training_histories = pickle.load(f)
else:
    all_training_histories = {}

if COMPLETION_STATUS_FILE.exists():
    with open(COMPLETION_STATUS_FILE, 'r', encoding='utf-8') as f:
        completion_status = json.load(f)
else:
    completion_status = {'completed_ratios': [], 'completed_years': {}}

summary_rows = []
for run_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[run_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    is_completed = run_key in completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'run_key': run_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('run_key').reset_index(drop=True) if summary_rows else pd.DataFrame()
print('Per-run training summary:')
display(summary_df)

combined_history_path = EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-run training summary:


,run_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,"RR_0.5_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)...",6,2022,0.175103,0.323007,True


Combined history file: mlp_classifier_history_prevyears_monthly_features_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)_all_ratios.csv
Rows: 6


,year,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,...,random_used_size,uncertain_threshold_mean,wmean_hard,wmean_confident,wmean_uncertain,wmean_positive_rate,wmean_random,artifact_suffix,run_key,replay_ratio
1,2018,0.874328,0.119734,0.805774,0.208488,0.867130,0.117413,0.767520,0.203669,0.898119,...,15597,0.871806,NaN,1.0,1.0,1.0,1.0,"_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)_RR=0.5","RR_0.5_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)...",0.5
2,2019,0.850826,0.101982,0.815523,0.181294,0.860089,0.095755,0.788108,0.170763,0.905687,...,15127,0.908764,NaN,1.0,1.0,1.0,1.0,"_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)_RR=0.5","RR_0.5_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)...",0.5
3,2020,0.887270,0.118743,0.786636,0.206339,0.893763,0.108231,0.729246,0.188488,0.901218,...,10029,0.862800,NaN,1.0,1.0,1.0,1.0,"_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)_RR=0.5","RR_0.5_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)...",0.5
4,2021,0.916464,0.126097,0.756767,0.216173,0.909441,0.116651,0.732616,0.201257,0.900570,...,7067,0.843060,NaN,1.0,1.0,1.0,1.0,"_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)_RR=0.5","RR_0.5_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)...",0.5
5,2022,0.786610,0.112854,0.869036,0.199766,0.781920,0.098267,0.802922,0.175103,0.876970,...,5724,0.836257,NaN,1.0,1.0,1.0,1.0,"_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)_RR=0.5","RR_0.5_combined_HE=0_CC=0.2_UP=0.1_PR=(0.2,10)...",0.5
